In [0]:
%sql
-- This should return 1 for every train. If any train has > 1, the MERGE failed and appended instead.
SELECT train_no, COUNT(*) as record_count 
FROM gold.dim_train_master_scd1 
GROUP BY train_no 
ORDER BY record_count DESC;



train_no,record_count
12309,1
12503,1
20504,1
12621,1
12649,1
19031,1
12951,1
22691,1
12002,1
12627,1


In [0]:
%sql
-- should see older rows with is_current = 'N' and fresh rows with is_current = 'Y'
SELECT train_no, eff_from_date, eff_to_date, is_current 
FROM gold.dim_train_schedule_scd2 
WHERE train_no = '12951'
ORDER BY eff_from_date DESC;

train_no,eff_from_date,eff_to_date,is_current
12951,2025-11-12,9999-12-31,Y
12951,2010-01-01,2025-11-11,N


In [0]:
%sql
-- Simulate a schedule change for a specific train
UPDATE bronze.live_train_status
SET scheduled_dep = '04:00' 
WHERE train_no = '12951';

num_affected_rows
2


In [0]:
from delta.tables import DeltaTable

# Grab the latest version number
delta_target = DeltaTable.forName(spark, 'gold.dim_train_schedule_scd2')
latest_version = int(delta_target.history(1).head().version)

# Stream the exact changes made during that specific version
spark.read.format('delta') \
    .option('readChangeFeed', 'true') \
    .option('startingVersion', latest_version) \
    .table('gold.dim_train_schedule_scd2') \
    .select("train_no", "_change_type", "eff_from_date", "is_current") \
    .show(20, truncate=False)

+--------+------------+-------------+----------+
|train_no|_change_type|eff_from_date|is_current|
+--------+------------+-------------+----------+
+--------+------------+-------------+----------+

